# RedSpell: data, training, evaluation
Pipeline: **1** data → **2** preprocessing → **3** representation → **4** modeling → **5** evaluation.

This notebook so far: setup (section 0) and the dataset (section 1).

## 0. Setup

In [ ]:
!pip -q install sacremoses symspellpy sentencepiece gdown
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE -> Runtime > Change runtime type > T4 GPU")

In [ ]:
import os, re, json, math, time, shutil
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import torch
import zipfile, gdown
from kaggle_secrets import UserSecretsClient

SEED = 42
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')      # saved as the notebook output (downloadable models, results)
TMP = Path('/kaggle/temp')          # scratch, NOT saved: data splits and checkpoints
DATA, CKPT, MODELS = TMP / 'data', TMP / 'ckpt', WORK / 'models'
for p in (DATA, CKPT, MODELS):
    p.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = WORK / 'results.json'

for z in INPUT.rglob('*.zip'):
    zipfile.ZipFile(z).extractall('/kaggle/temp/lang8')

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -> Settings > Accelerator > GPU')
RAW = TMP / 'lang8'
RAW.mkdir(parents=True, exist_ok=True)
if not any(RAW.rglob('entries.train')):
    fid = UserSecretsClient().get_secret('LANG8_FOLDER_ID')
    gdown.download_folder(url=f'https://drive.google.com/drive/folders/{fid}', output=str(RAW), quiet=True, use_cookies=False)
train_file = next(RAW.rglob('entries.train'), None)
assert train_file, 'entries.train not found after download: check the folder ID, the sharing setting and Internet=On'
LANG8_EN_DIR = train_file.parent
print(sorted(p.name for p in LANG8_EN_DIR.iterdir()))

In [ ]:
REAL_TRAIN_TSV = DATA / 'real_train_pairs.tsv'   # source<TAB>target, built in section 1.1
REAL_TEST_TSV = DATA / 'real_test_pairs.tsv'     # official entries.test split
N_TRAIN = 100_000       # training pairs to keep (cap for training time)
N_VAL = 3_000           # validation pairs held out from train

RUN_SMOKE = True        # 2-minute check of the training loop before the long runs
RUN_SMALL = True        # t5-small, about 30 min
RUN_BASE = True         # t5-base, about 90 min
SHOW_SAMPLES = False    # True prints real corpus sentences: only with a private notebook

## 1. Dataset: Lang-8 English corpus v1.0
### 1.1 Inspect and parse
Format (from its README): tab-separated, one learner sentence per line; columns = number of corrections, serial number, URL, sentence number (0 = journal title), learner sentence, then the correction(s). The preview masks all letters and digits, so no corpus text appears in the output.

In [ ]:
def parse_entries(path, out_path):
    n_lines = n_pairs = n_title = 0
    with open(path, encoding='utf-8') as f, open(out_path, 'w', encoding='utf-8') as out:
        for line in f:
            cols = line.rstrip('\n').split('\t')
            if len(cols) < 6:
                continue                          # blank separator, or no correction
            n_lines += 1
            if cols[3] == '0':                    # sentence number 0 = journal title
                n_title += 1
                continue
            src, tgt = cols[4].strip(), cols[5].strip()   # first correction only
            if src and tgt:
                out.write(f'{src}\t{tgt}\n'); n_pairs += 1
    print(f'{path.name}: {n_lines} corrected lines | {n_title} titles skipped | {n_pairs} pairs')

parse_entries(LANG8_EN_DIR / 'entries.train', REAL_TRAIN_TSV)
parse_entries(LANG8_EN_DIR / 'entries.test', REAL_TEST_TSV)

### 1.2 Clean and split
Detokenize, filter, remove any training sentence that also appears in the test set (no leakage), hold out a validation set. Saves `train.jsonl`, `val.jsonl`, `test.jsonl`.

In [ ]:
from sacremoses import MosesDetokenizer

md_ = MosesDetokenizer(lang='en')
STANFORD = [('-LRB-', '('), ('-RRB-', ')'), ('-LSB-', '['), ('-RSB-', ']'), ('``', '"'), ("''", '"')]

def detok(s):
    for a, b in STANFORD:
        s = s.replace(a, b)
    return re.sub(r" n't\b", "n't", md_.detokenize(s.split()))

def prep(path):
    d = pd.read_csv(path, sep='\t', header=None, names=['source', 'target'],
                    quoting=3, on_bad_lines='skip', dtype=str).dropna()
    d = d[d.source.str.strip().ne('') & d.target.str.strip().ne('')]
    # v1.0 is Stanford-tokenized ('did n't ... !'); the app gets natural text, so detokenize
    probe = d.source.sample(min(len(d), 2000), random_state=0)
    if probe.str.contains(r" n't| [.,!?]$").mean() > 0.3:
        d['source'], d['target'] = d.source.map(detok), d.target.map(detok)
    d = d[d.source.map(str.isascii) & d.target.map(str.isascii)]
    wc = lambda s: s.str.split().str.len()
    d = d[wc(d.source).between(3, 50) & wc(d.target).between(3, 50)]
    d = d[(wc(d.target) / wc(d.source)).between(0.5, 2.0)]        # drop misaligned pairs
    return d.drop_duplicates('source').reset_index(drop=True)

tr = prep(REAL_TRAIN_TSV).sample(frac=1, random_state=SEED).reset_index(drop=True)
te = prep(REAL_TEST_TSV)
tr = tr[~tr.source.isin(set(te.source))].reset_index(drop=True)   # no train/test overlap
n_val = min(N_VAL, len(tr) // 10)
va, tr = tr[:n_val], tr[n_val:n_val + N_TRAIN]
for name, part in {'train': tr, 'val': va, 'test': te}.items():
    part.to_json(DATA / f'{name}.jsonl', orient='records', lines=True)
    print(name, len(part))
print('unchanged pairs (train): %.1f%%' % (100 * (tr.source == tr.target).mean()))
if SHOW_SAMPLES:
    display(tr.sample(8, random_state=1))

## 2. Representation: subword tokenization
T5 uses a SentencePiece subword vocabulary, so typos and unseen words are split into known pieces instead of becoming `<unk>`. This cell shows it, measures sequence lengths to set `MAX_LEN`, and drops training pairs that contain characters the T5 vocabulary lacks.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup

MODEL_NAME = 't5-small'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

train = pd.read_json(DATA / 'train.jsonl', lines=True, dtype=False)
val = pd.read_json(DATA / 'val.jsonl', lines=True, dtype=False)
test = pd.read_json(DATA / 'test.jsonl', lines=True, dtype=False)

# subword behaviour on typos (invented example, not corpus text)
print(tok.tokenize('I recieved teh pakage yesterday .'))

ids = lambda texts: tok(list(texts), add_special_tokens=True)['input_ids']
smp = train.sample(min(20_000, len(train)), random_state=SEED)
lens = np.array([len(x) for x in ids(smp.source)] + [len(x) for x in ids(smp.target)])
MAX_LEN = int(np.percentile(lens, 99.5))
print(f'tokens per sentence: mean {lens.mean():.1f}, p99.5 {MAX_LEN}, max {lens.max()} -> MAX_LEN = {MAX_LEN}')

def has_unk(texts):
    return np.array([tok.unk_token_id in x for x in tok(list(texts), add_special_tokens=False)['input_ids']])

for name in ('train', 'val'):
    d = globals()[name]
    bad = has_unk(d.source) | has_unk(d.target)
    print(f'{name}: dropping {bad.sum()} pairs with out-of-vocabulary characters ({100 * bad.mean():.2f}%)')
    globals()[name] = d[~bad].reset_index(drop=True)   # test stays untouched
print(len(train), len(val), len(test))

## 3. Metric and baselines
**GLEU** as used for grammatical error correction (Napoles et al., 2016): n-gram overlap with the reference, penalizing n-grams copied from the source that are absent from the reference. NLTK's `gleu_score` is a different metric (it ignores the source), so it is implemented here. Baselines: copying the input, and a spelling-only SymSpell corrector.

In [ ]:
_tokens = lambda s: re.findall(r"\w+|[^\w\s]", s)
_ngrams = lambda t, n: Counter(tuple(t[i:i + n]) for i in range(len(t) - n + 1))

def gleu_corpus(sources, hyps, refs, max_n=4):
    """Corpus-level GLEU (Napoles et al. 2016), one reference per sentence.
    p_n = (N(H,R) - [N(H,S) - N(H,S,R)]) / N(H), N(A,B) = sum_g min(count_A(g), count_B(g))."""
    num, den, hl, rl = [0.0] * max_n, [0] * max_n, 0, 0
    for s, h, r in zip(sources, hyps, refs):
        s, h, r = _tokens(s), _tokens(h), _tokens(r)
        hl += len(h); rl += len(r)
        for n in range(1, max_n + 1):
            H, S, R = _ngrams(h, n), _ngrams(s, n), _ngrams(r, n)
            num[n - 1] += max(sum((H & R).values()) - (sum((H & S).values()) - sum((H & S & R).values())), 0)
            den[n - 1] += sum(H.values())
    if hl == 0 or any(d == 0 for d in den):
        return 0.0
    bp = 1.0 if hl > rl else math.exp(1 - rl / hl)
    return bp * math.exp(sum(math.log(max(a, 1e-9) / d) for a, d in zip(num, den)) / max_n)

def evaluate(sources, hyps, refs):
    sources, hyps, refs = list(sources), list(hyps), list(refs)
    return {'GLEU': round(100 * gleu_corpus(sources, hyps, refs), 2),
            'exact_match_%': round(100 * np.mean([h == r for h, r in zip(hyps, refs)]), 2),
            'changed_%': round(100 * np.mean([h != s for h, s in zip(hyps, sources)]), 2)}

_s = ['I has a apple .', 'She go to school .']; _r = ['I have an apple .', 'She goes to school .']
print('perfect:', evaluate(_s, _r, _r)['GLEU'], '| copy:', evaluate(_s, _s, _r)['GLEU'])

In [ ]:
import importlib.resources as ir
from symspellpy import SymSpell, Verbosity

sym = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
sym.load_dictionary(str(ir.files('symspellpy') / 'frequency_dictionary_en_82_765.txt'), term_index=0, count_index=1)
_TOK = re.compile(r'[A-Za-z]+|[^A-Za-z]+')

def symspell_correct(s):
    """Word-by-word spelling fix; keeps punctuation and casing, skips capitalized mid-sentence words (names)."""
    out, first = [], True
    for t in _TOK.findall(s):
        if t[0].isalpha() and len(t) > 3 and not (t[0].isupper() and not first):
            top = sym.lookup(t.lower(), Verbosity.TOP, max_edit_distance=2)
            if top and top[0].term != t.lower():
                w = top[0].term
                t = w.upper() if t.isupper() and len(t) > 1 else (w.capitalize() if t[0].isupper() else w)
        first = first and not t[0].isalpha()
        out.append(t)
    return ''.join(out)

results = {}
results['copy source'] = evaluate(test.source, test.source, test.target)
results['SymSpell (spelling only)'] = evaluate(test.source, [symspell_correct(s) for s in test.source], test.target)
json.dump(results, open(RESULTS_PATH, 'w'), indent=1)
pd.DataFrame(results).T

## 4. Modeling: fine-tune T5
Input `grammar: <incorrect sentence>`, target the corrected sentence. Plain PyTorch loop, **model choice on validation GLEU, never on the test set**. Checkpoints go to `/kaggle/temp` (not saved in the output); if a session dies, re-run the cells above and the training cell resumes.

In [ ]:
assert torch.cuda.is_available(), 'Need a GPU: Settings > Accelerator > GPU'
DEVICE = 'cuda'
PREFIX = 'grammar: '            # T5-style task prefix
SRC_LEN = MAX_LEN + 4           # room for the prefix tokens
DEMO = ['She go to school every days .', 'I recieved teh pakage yesterday .', "He don't likes apples ."]  # invented

def encode(df, tokenizer):
    x = tokenizer([PREFIX + s for s in df.source], max_length=SRC_LEN, truncation=True)['input_ids']
    y = tokenizer(text_target=list(df.target), max_length=MAX_LEN, truncation=True)['input_ids']
    return x, y

def collate(xs, ys, pad_id):
    lx, ly = max(map(len, xs)), max(map(len, ys))
    inp = torch.full((len(xs), lx), pad_id, dtype=torch.long)
    att = torch.zeros((len(xs), lx), dtype=torch.long)
    lab = torch.full((len(ys), ly), -100, dtype=torch.long)
    for i, (x, y) in enumerate(zip(xs, ys)):
        inp[i, :len(x)] = torch.tensor(x); att[i, :len(x)] = 1; lab[i, :len(y)] = torch.tensor(y)
    return inp, att, lab

@torch.no_grad()
def correct(model, tokenizer, sentences, batch_size=64, num_beams=4):
    model.eval()
    order = sorted(range(len(sentences)), key=lambda i: len(sentences[i]))   # similar lengths -> less padding
    out = [None] * len(sentences)
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        enc = tokenizer([PREFIX + sentences[j] for j in idx], return_tensors='pt', padding=True,
                        truncation=True, max_length=SRC_LEN).to(DEVICE)
        gen = model.generate(**enc, num_beams=num_beams, max_new_tokens=MAX_LEN)
        for j, t in zip(idx, tokenizer.batch_decode(gen, skip_special_tokens=True)):
            out[j] = t
    return out

def save_ckpt(path, model, opt, sched, state):
    tmp = str(path) + '.tmp'                       # write then rename: a crash cannot leave a half file
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                'sched': sched.state_dict(), 'state': state}, tmp)
    os.replace(tmp, path)

def run_training(model_name, tag, epochs=3, lr=3e-4, batch_size=32, eval_n=1000, fp16=False, smoke=False, ckpt_every=1000):
    """Fine-tune `model_name`; best model (by validation GLEU) goes to MODELS/<tag>."""
    torch.manual_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_df = train.head(2000) if smoke else train
    xs, ys = encode(train_df, tokenizer)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps_per_epoch = len(xs) // batch_size
    total = steps_per_epoch * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.03 * total), total)
    scaler = torch.amp.GradScaler('cuda', enabled=fp16)

    state = {'epoch': 0, 'step_in_epoch': 0, 'best_gleu': -1.0, 'log': []}
    ckpt = CKPT / f'{tag}_last.pt'
    if ckpt.exists() and not smoke:
        ck = torch.load(ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        state = ck['state']
        print('resumed from epoch', state['epoch'] + 1, 'step', state['step_in_epoch'])

    val_sub = val.sample(min(100 if smoke else eval_n, len(val)), random_state=SEED)
    print('val reference (copy source) GLEU:', evaluate(val_sub.source, val_sub.source, val_sub.target)['GLEU'])
    print(f'{len(xs)} pairs | {steps_per_epoch} steps/epoch | {epochs} epochs')

    for epoch in range(state['epoch'], epochs):
        perm = np.random.RandomState(SEED + epoch).permutation(len(xs))   # same order after a resume
        model.train(); t0 = time.time()
        for step in range(state['step_in_epoch'], steps_per_epoch):
            idx = perm[step * batch_size:(step + 1) * batch_size]
            inp, att, lab = (t.to(DEVICE) for t in collate([xs[i] for i in idx], [ys[i] for i in idx], tokenizer.pad_token_id))
            with torch.autocast('cuda', dtype=torch.float16, enabled=fp16):
                loss = model(input_ids=inp, attention_mask=att, labels=lab).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            if step % 200 == 0:
                print(f'epoch {epoch + 1} step {step}/{steps_per_epoch} loss {loss.item():.4f} ({time.time() - t0:.0f}s)')
            if not smoke and (step + 1) % ckpt_every == 0:
                save_ckpt(ckpt, model, opt, sched, dict(state, epoch=epoch, step_in_epoch=step + 1))

        state['epoch'], state['step_in_epoch'] = epoch + 1, 0
        m = evaluate(val_sub.source, correct(model, tokenizer, list(val_sub.source)), val_sub.target)
        m.update(epoch=epoch + 1, minutes=round((time.time() - t0) / 60, 1))
        state['log'].append(m)
        print('VAL', m)
        print(correct(model, tokenizer, DEMO))
        if m['GLEU'] > state['best_gleu']:
            state['best_gleu'] = m['GLEU']
            model.save_pretrained(MODELS / tag); tokenizer.save_pretrained(MODELS / tag)
        if not smoke:
            save_ckpt(ckpt, model, opt, sched, state)
    return pd.DataFrame(state['log'])

def eval_test(tag, name=None):
    """Test-set evaluation of MODELS/<tag>; adds a row to `results` (saved in the output)."""
    tokenizer = AutoTokenizer.from_pretrained(MODELS / tag)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODELS / tag).to(DEVICE)
    hyp = correct(model, tokenizer, list(test.source))
    results[name or tag] = evaluate(test.source, hyp, test.target)
    json.dump(results, open(RESULTS_PATH, 'w'), indent=1)
    pd.DataFrame({'source': test.source, 'target': test.target, 'prediction': hyp}).to_json(
        DATA / f'{tag}_test_predictions.jsonl', orient='records', lines=True)   # in /kaggle/temp: not saved
    del model; torch.cuda.empty_cache()
    return hyp

### 4.1 Smoke test
Whole loop on 2,000 pairs for one epoch (about 2 minutes): catches bugs before the long runs.

In [ ]:
if RUN_SMOKE:
    run_training(MODEL_NAME, 'smoke', epochs=1, smoke=True)
    shutil.rmtree(MODELS / 'smoke', ignore_errors=True)   # do not keep it in the output

### 4.2 t5-small
3 epochs, batch 32, learning rate 3e-4, fp32 (T5 can produce NaN losses in fp16).

In [ ]:
if RUN_SMALL:
    log_small = run_training('t5-small', 't5-small', epochs=3, lr=3e-4, batch_size=32)
    display(log_small)
    hyp_small = eval_test('t5-small')

### 4.3 t5-base
Lower learning rate and smaller batch than `t5-small`. If you get a CUDA out-of-memory error, set `batch_size=8`.

In [ ]:
if RUN_BASE:
    log_base = run_training('t5-base', 't5-base', epochs=2, lr=1e-4, batch_size=16, ckpt_every=2000)
    display(log_base)
    hyp_base = eval_test('t5-base')

## 5. Results
The table below is also saved as `results.json`. Download the models from the notebook's **Output** tab (`models/t5-base`, `models/t5-small`).

In [ ]:
print(pd.DataFrame(results).T)
for p in sorted(MODELS.glob('*')):
    print(p.name, '%.0f MB' % (sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e6))

| Model | Test GLEU | Exact match % | Sentences changed % |
|---|---|---|---|
| Copy the input (floor) | 35.43 | 2.04 | 0.00 |
| SymSpell (spelling only) | 37.46 | 3.53 | 18.51 |
| `t5-small` (3 epochs) | 47.81 | 12.47 | 65.02 |
| **`t5-base` (2 epochs)** | **51.09** | **15.89** | 70.94